# VibeScent Week 5 — Qwen3-VL Unified Stack

**Owner:** Harsh Agarwal  
**Pipeline version:** `w5v1`  
**GPU:** Blackwell 6000 / A100 80 GB (bfloat16, no quantisation needed)  

## What changed from Week 4
| | Week 4 | Week 5 |
|---|---|---|
| Text embedder | Qwen3-Embedding-8B (separate) | Qwen3-VL-Embedding-8B (unified) |
| Multimodal embedder | Qwen3-VL-Embedding-8B | same model — no split |
| Corpus space | Qwen3-Embedding-8B space | Qwen3-VL-Embedding-8B space |
| Reranker | Gemini API (text-only) | Qwen3-VL-Reranker-8B (sees outfit image) |
| API calls at inference | Gemini Vision + Gemini Embed | **Zero** |
| VRAM used | ~34 GB (two 8B models) | ~34 GB (embedder + reranker) |

## Stage Map
| Stage | Description |
|-------|-------------|
| 0 | **ONE-TIME** — Re-embed corpus with Qwen3-VL-Embedding-8B, save to Drive |
| 1 | Setup — clone, deps, Drive, secrets, GPU detect |
| 2 | Load Artifacts — new Qwen3VL corpus embeddings + DataFrame |
| 3 | Load Models — Qwen3-VL-Embedding-8B + Qwen3-VL-Reranker-8B |
| 4 | Build Engine — VibeScentEngine (unified Qwen3VL) |
| 5 | Start FastAPI |
| 6 | Dual Tunnel — Cloudflare + ngrok |
| 7 | Demo Cache — 5 locked responses |
| 8 | Frontend Patch diff |
| 9 | Smoke Test |
| 10 | Pre-Demo Runbook |

## Pre-Demo Checklist
- [ ] Colab runtime: **A100 80 GB** or Blackwell 6000 selected
- [ ] Google Drive mounted, `vibescent/` visible at `DRIVE_BASE`
- [ ] **Run Stage 0 ONCE** → `qwen3vl_corpus/embeddings.npy` on Drive
- [ ] Then skip Stage 0 on subsequent runs
- [ ] No API keys needed at inference — HuggingFace token only (for model download)
- [ ] `HF_TOKEN` stored in Colab Secrets

In [ ]:
# ============================================================
# STAGE 0 — ONE-TIME corpus re-embedding with Qwen3-VL-Embedding-8B
# Run this cell ONCE. It saves qwen3vl_corpus/embeddings.npy to Drive.
# Skip on subsequent runs — Stage 2 loads the saved file.
# Runtime: ~8 min on A100 80GB (35k rows, batch 64)
# ============================================================
import os, sys, subprocess, time, glob
import traceback
import IPython

def custom_exc(shell, etype, evalue, tb, tb_offset=None):
    print("\n" + "="*60)
    print("!!! AN ERROR OCCURRED !!!")
    print("="*60)
    traceback.print_exception(etype, evalue, tb)
    print("="*60 + "\n")

_ipython = IPython.get_ipython()
if _ipython:
    _ipython.set_custom_exc((Exception,), custom_exc)
    _ipython.magic("xmode Verbose")

REPO_DIR = '/content/vibescent'
ZIP_PATH = '/content/drive/MyDrive/vibescent.zip'

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

if not os.path.exists(REPO_DIR):
    if os.path.exists(ZIP_PATH):
        print(f"Extracting {ZIP_PATH} to /content/...")
        subprocess.check_call(['unzip', '-q', ZIP_PATH, '-d', '/content/'])
        extracted_dirs = [d for d in os.listdir('/content')
                         if os.path.isdir(os.path.join('/content', d))
                         and 'vibescent' in d.lower()]
        if extracted_dirs and not os.path.exists(REPO_DIR):
            found_dir = os.path.join('/content', extracted_dirs[0])
            if found_dir != REPO_DIR:
                os.rename(found_dir, REPO_DIR)
        print(f"Successfully extracted to {REPO_DIR}")
    else:
        raise FileNotFoundError(ZIP_PATH)
else:
    print(f"Project folder already exists at {REPO_DIR}. Skipping extraction.")

os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'],
               check=True, capture_output=True)

_deps = ['torch>=2.4,<3.0', 'transformers>=4.57,<5.0', 'accelerate>=1.3,<2.0',
         'qwen-vl-utils>=0.0.14', 'Pillow>=10.0', 'numpy', 'pandas', 'tqdm']
subprocess.run(['uv', 'pip', 'install', '--system', '-q', '-e', REPO_DIR] + _deps,
               check=True, capture_output=True)

DRIVE_BASE = '/content/drive/MyDrive/vibescent'
OUT_DIR    = f'{DRIVE_BASE}/qwen3vl_corpus'
os.makedirs(OUT_DIR, exist_ok=True)

OUT_EMB  = f'{OUT_DIR}/embeddings.npy'
OUT_META = f'{OUT_DIR}/metadata_path.txt'

if os.path.exists(OUT_EMB):
    import numpy as np
    _existing = np.load(OUT_EMB)
    print(f'[SKIP] Corpus already embedded: {OUT_EMB}  shape={_existing.shape}')
    print('Stage 0 complete — proceed to Stage 1.')
else:
    import torch, numpy as np, pandas as pd
    from tqdm.auto import tqdm
    from pathlib import Path

    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
    from vibescents.enrich import build_retrieval_text

    _csv_cands = [
        f'{DRIVE_BASE}/vibescent_enriched_2k_v2.csv',
        f'{DRIVE_BASE}/vibescent_enriched_500_v2.csv',
        f'{REPO_DIR}/data/processed/vibescent_unified.csv',
        f'{REPO_DIR}/data/vibescent_enriched.csv',
    ]
    df = None
    for _p in _csv_cands:
        if os.path.exists(_p):
            df = pd.read_csv(_p)
            print(f'DataFrame: {_p}  shape={df.shape}')
            break
    if df is None:
        raise FileNotFoundError('No fragrance DataFrame found. Run Week 2 first.')

    # Use the canonical build_retrieval_text() so embedding texts match the offline pipeline
    df = build_retrieval_text(df)
    texts = df['retrieval_text'].fillna(df.get('name', '')).tolist()
    print(f'Built {len(texts)} retrieval texts. Sample: {texts[0][:100]}')

    from vibescents.embeddings import Qwen3VLMultimodalEmbedder
    from vibescents.settings import Settings

    s = Settings()
    Qwen3VLMultimodalEmbedder._BATCH_SIZE = 64
    embedder = Qwen3VLMultimodalEmbedder(settings=s, load_in_8bit=False)
    print('Model loaded. Starting corpus embedding...')

    CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
    os.makedirs(CKPT_DIR, exist_ok=True)

    def get_checkpoints():
        return sorted(glob.glob(os.path.join(CKPT_DIR, "embeddings_ckpt_*.npy")),
                      key=lambda p: int(Path(p).stem.split("_")[-1]))

    already_embedded = sum(np.load(f).shape[0] for f in get_checkpoints())
    print(f'Already embedded: {already_embedded} / {len(texts)}')
    texts_to_embed = texts[already_embedded:]

    if texts_to_embed:
        t0 = time.perf_counter()
        CHUNK_SIZE = Qwen3VLMultimodalEmbedder._BATCH_SIZE * 50
        with tqdm(total=len(texts_to_embed), desc="Embedding (Outer Chunks)") as outer_pbar:
            for i in range(0, len(texts_to_embed), CHUNK_SIZE):
                chunk = texts_to_embed[i:i+CHUNK_SIZE]
                chunk_emb = embedder.embed_multimodal_documents(chunk)
                ckpt_path = os.path.join(CKPT_DIR, f"embeddings_ckpt_{len(get_checkpoints())}.npy")
                np.save(ckpt_path, chunk_emb)
                outer_pbar.update(len(chunk))
        print(f'Embedded {len(texts_to_embed)} rows in {time.perf_counter()-t0:.0f}s')

    corpus_emb = np.vstack([np.load(f) for f in get_checkpoints()])
    np.save(OUT_EMB, corpus_emb.astype(np.float32))
    with open(OUT_META, 'w') as _mf:
        _mf.write(str(_csv_cands[0]))

    _local_out = f'{REPO_DIR}/artifacts/qwen3vl_corpus'
    os.makedirs(_local_out, exist_ok=True)
    np.save(f'{_local_out}/embeddings.npy', corpus_emb.astype(np.float32))

    print(f'Saved to: {OUT_EMB}')
    print(f'Shape: {corpus_emb.shape} | dtype: {corpus_emb.dtype}')
    print('Stage 0 COMPLETE. Proceed to Stage 1.')


In [ ]:
# Stage 1: Setup
import os, sys, subprocess, time, traceback
_t0 = time.perf_counter()
PIPELINE_VERSION = 'w5v1'
REPO_DIR  = '/content/vibescent'
FASTAPI_PORT = 8000; FASTAPI_HOST = '127.0.0.1'
_BASE_URL = f'http://{FASTAPI_HOST}:{FASTAPI_PORT}'

import IPython

def custom_exc(shell, etype, evalue, tb, tb_offset=None):
    print("\n" + "="*60)
    print("!!! AN ERROR OCCURRED !!!")
    print("="*60)
    traceback.print_exception(etype, evalue, tb)
    print("="*60 + "\n")

_ipython = IPython.get_ipython()
if _ipython:
    _ipython.set_custom_exc((Exception,), custom_exc)
    _ipython.magic("xmode Verbose")

print("Step 1: Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ZIP_CANDS = [
    '/content/drive/MyDrive/vibescent.zip',
    '/content/drive/My Drive/vibescent.zip',
    '/content/vibescent.zip'
]
ZIP_PATH = next((c for c in ZIP_CANDS if os.path.exists(c)), None)

if not os.path.isdir(REPO_DIR):
    if ZIP_PATH:
        print(f"Step 2: Extracting {ZIP_PATH} to /content/...")
        subprocess.check_call(['unzip', '-q', ZIP_PATH, '-d', '/content/'])
        extracted_dirs = [d for d in os.listdir('/content')
                         if os.path.isdir(os.path.join('/content', d))
                         and 'vibescent' in d.lower()]
        if extracted_dirs and not os.path.exists(REPO_DIR):
            found_dir = os.path.join('/content', extracted_dirs[0])
            if found_dir != REPO_DIR:
                os.rename(found_dir, REPO_DIR)
    else:
        print("\n[CRITICAL ERROR] vibescent.zip NOT FOUND.")
        raise FileNotFoundError("Could not find vibescent.zip in Drive or local storage.")
else:
    print('Project folder already exists. Skipping extraction.')

os.chdir(REPO_DIR)

# ── Drive paths ───────────────────────────────────────────────────────────
DRIVE_BASE       = '/content/drive/MyDrive/vibescent'
HF_CACHE         = '/content/drive/MyDrive/hf_cache'
W5_ARTIFACTS     = f'{DRIVE_BASE}/week5'
CLOUDFLARED_BIN  = '/usr/local/bin/cloudflared'

for _d in [DRIVE_BASE, HF_CACHE, W5_ARTIFACTS]:
    os.makedirs(_d, exist_ok=True)

os.environ['HF_HOME']                = HF_CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# ── HF token — stored in Colab Secrets (sidebar → 🔑 Secrets → HF_TOKEN) ──
try:
    from google.colab import userdata as _ud
    _hf_token = _ud.get('HF_TOKEN') or ''
except Exception:
    _hf_token = os.environ.get('HF_TOKEN', '')

if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab Secrets.')
else:
    print('[WARNING] HF_TOKEN not found. Add it via: sidebar → Secrets → + New secret: HF_TOKEN')
    print('         First-time model download will fail without it.')

# ── UV cache + package install ────────────────────────────────────────────
UV_CACHE = '/content/drive/MyDrive/uv_cache'
os.makedirs(UV_CACHE, exist_ok=True)
os.environ['UV_CACHE_DIR'] = UV_CACHE

print("Step 3: Installing 'uv'...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'],
               check=True, capture_output=True)

from tqdm.auto import tqdm
bar = tqdm(['Pkg', 'Deps', 'Cloudflared', 'GPU'], desc='Stage 1', ncols=100, unit='step')

subprocess.run(['uv', 'pip', 'install', '--system', '-q', '-e', REPO_DIR],
               check=True, capture_output=True)
bar.update(1)

# Note: vllm is NOT listed — the reranker uses AutoModelForCausalLM (standard transformers).
#       outlines/jedi are offline-enrichment deps; not needed for inference.
_DEPS = [
    'torch>=2.4,<3.0', 'transformers>=4.57,<5.0', 'accelerate>=1.3,<2.0',
    'bitsandbytes',
    'qwen-vl-utils>=0.0.14', 'fastapi>=0.115,<1.0', 'uvicorn[standard]>=0.34,<1.0',
    'httpx>=0.27,<1.0', 'pyngrok>=7.2,<8.0', 'nest-asyncio>=1.6,<2.0',
    'Pillow>=10.0,<12.0', 'tqdm', 'pandas', 'numpy',
]
subprocess.run(['uv', 'pip', 'install', '--system', '-q'] + _DEPS,
               check=True, capture_output=True)
bar.update(1)

if not os.path.isfile(CLOUDFLARED_BIN):
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', CLOUDFLARED_BIN], capture_output=True)
    subprocess.run(['chmod', '+x', CLOUDFLARED_BIN], capture_output=True)
bar.update(1)

import torch
GPU_TIER = 'CPU'; DEVICE = 'cpu'
if torch.cuda.is_available():
    DEVICE = 'cuda'
    _v = torch.cuda.get_device_properties(0).total_memory / 1e9
    GPU_TIER = ('BLACKWELL' if _v >= 75 else ('A100' if _v >= 35 else ('L4' if _v >= 20 else 'T4')))
    print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM={_v:.1f}GB  tier={GPU_TIER}')
else:
    print('[WARNING] No GPU detected — inference will be very slow.')

bar.update(1); bar.close()
print(f'Setup {time.perf_counter()-_t0:.1f}s | DRIVE={DRIVE_BASE} | GPU={GPU_TIER}')


In [ ]:
# Stage 2: Load Qwen3VL corpus embeddings + DataFrame
import time, os
import numpy as np
import pandas as pd
_t0 = time.perf_counter()

# ── Corpus embeddings (Qwen3-VL-Embedding-8B space) ──────────────────────────
_EMB_CANDS = [
    f'{DRIVE_BASE}/qwen3vl_corpus/embeddings.npy',          # Stage 0 output on Drive
    f'{REPO_DIR}/artifacts/qwen3vl_corpus/embeddings.npy',  # local fallback
]
CORPUS_EMB_PATH = None
for _p in _EMB_CANDS:
    if os.path.exists(_p):
        CORPUS_EMB_PATH = _p
        break
if CORPUS_EMB_PATH is None:
    raise FileNotFoundError(
        'Qwen3VL corpus embeddings not found.\n'
        'Run Stage 0 first to generate them.\n'
        f'Expected at: {_EMB_CANDS[0]}'
    )

print(f'Loading corpus embeddings from {CORPUS_EMB_PATH} ...', end=' ', flush=True)
CORPUS_EMBEDDINGS = np.load(CORPUS_EMB_PATH).astype(np.float32)
# L2-normalise for cosine similarity via dot product
_norms = np.linalg.norm(CORPUS_EMBEDDINGS, axis=1, keepdims=True)
CORPUS_EMBEDDINGS = CORPUS_EMBEDDINGS / np.where(_norms < 1e-9, 1.0, _norms)
print(f'done. shape={CORPUS_EMBEDDINGS.shape}')

# ── Fragrance DataFrame ───────────────────────────────────────────────────────
_CSV_CANDS = [
    f'{DRIVE_BASE}/vibescent_enriched_2k_v2.csv',
    f'{DRIVE_BASE}/vibescent_enriched_500_v2.csv',
    f'{REPO_DIR}/data/processed/vibescent_unified.csv',
    f'{REPO_DIR}/data/vibescent_enriched.csv',
]
FRAGRANCE_DF = None
for _c in _CSV_CANDS:
    if os.path.exists(_c):
        FRAGRANCE_DF = pd.read_csv(_c)
        print(f'DataFrame: {_c}  shape={FRAGRANCE_DF.shape}')
        break
if FRAGRANCE_DF is None:
    raise FileNotFoundError('No fragrance DataFrame found. Run Week 2 first.')

# Align lengths
if len(CORPUS_EMBEDDINGS) != len(FRAGRANCE_DF):
    raise AssertionError(
        f'Embedding/DataFrame length mismatch: '
        f'emb={len(CORPUS_EMBEDDINGS)} vs df={len(FRAGRANCE_DF)}.\n'
        f'Re-run Stage 0 (or harsh_offline_pipeline.ipynb Stage 5) to regenerate '
        f'corpus embeddings from the current DataFrame.'
    )

if 'fragrance_id' not in FRAGRANCE_DF.columns:
    FRAGRANCE_DF.insert(0, 'fragrance_id', FRAGRANCE_DF.index.astype(str))

print(f'\nArtifacts ready in {time.perf_counter()-_t0:.1f}s')
print(f'  Corpus: {CORPUS_EMBEDDINGS.shape}  (Qwen3-VL-Embedding-8B space, L2-normed)')
print(f'  DataFrame: {FRAGRANCE_DF.shape}')

In [ ]:
# Stage 3: Load Qwen3-VL-Embedding-8B + Qwen3-VL-Reranker-8B
# Both in bfloat16. Total VRAM: ~34 GB. No quantisation needed on 80 GB GPU.
import time, torch, gc
from tqdm.auto import tqdm
import sys; sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
_t0 = time.perf_counter()

gc.collect(); torch.cuda.empty_cache()

# TF32: same dynamic range as float32, 10 fewer mantissa bits, ~30% faster matmuls
# Safe on Blackwell/Ampere — enabled by default since PyTorch 1.12 but be explicit
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── 1. Qwen3-VL-Embedding-8B ─────────────────────────────────────────────────
EMBEDDER_MODEL = 'Qwen/Qwen3-VL-Embedding-8B'
print(f'Loading {EMBEDDER_MODEL} (bfloat16) ...')
from vibescents.embeddings import Qwen3VLMultimodalEmbedder
from vibescents.settings import Settings

_s = Settings()
Qwen3VLMultimodalEmbedder._BATCH_SIZE = 8  # conservative for inference
VL_EMBEDDER = Qwen3VLMultimodalEmbedder(settings=_s, load_in_8bit=False)

_warmup = VL_EMBEDDER.embed_multimodal_documents(['warmup fragrance'])
EMB_DIM = _warmup.shape[1]
print(f'Embedder ready. dim={EMB_DIM}  VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

assert EMB_DIM == CORPUS_EMBEDDINGS.shape[1], (
    f'Embedding dimension mismatch: model={EMB_DIM} vs corpus={CORPUS_EMBEDDINGS.shape[1]}.\n'
    'Re-run Stage 0 (or harsh_offline_pipeline.ipynb) to regenerate corpus embeddings.'
)
print(f'Dimension check passed: {EMB_DIM}')

# ── 2. Qwen3-VL-Reranker-8B ──────────────────────────────────────────────────
RERANKER_MODEL = 'Qwen/Qwen3-VL-Reranker-8B'
print(f'\nLoading {RERANKER_MODEL} (bfloat16, flash_attention_2) ...')

from transformers import AutoProcessor, AutoModelForCausalLM

VL_RERANKER_PROC = AutoProcessor.from_pretrained(
    RERANKER_MODEL, trust_remote_code=True
)
VL_RERANKER_MODEL = AutoModelForCausalLM.from_pretrained(
    RERANKER_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='cuda',
    trust_remote_code=True,
    attn_implementation='flash_attention_2',
).eval()

# torch.compile: fuses ops, reduces kernel-launch overhead
# mode='reduce-overhead' is safe with dynamic shapes (padding varies per batch)
try:
    VL_RERANKER_MODEL = torch.compile(
        VL_RERANKER_MODEL, mode='reduce-overhead', fullgraph=False
    )
    print('torch.compile applied to reranker')
except Exception as _ce:
    print(f'torch.compile skipped ({_ce}) — running uncompiled')

_tok = VL_RERANKER_PROC.tokenizer
_YES_IDS = _tok.encode('yes', add_special_tokens=False)
_NO_IDS  = _tok.encode('no',  add_special_tokens=False)
_YES_ID  = _YES_IDS[0] if _YES_IDS else -1
_NO_ID   = _NO_IDS[0]  if _NO_IDS  else -1

print(f'Reranker ready. yes_id={_YES_ID} no_id={_NO_ID}')
# ── GPU corpus for O(1) retrieval matmul ─────────────────────────────────
print('Moving corpus embeddings to GPU...')
CORPUS_EMBEDDINGS_GPU = torch.from_numpy(CORPUS_EMBEDDINGS).to('cuda', non_blocking=True)
# float32 retained for numerical accuracy; 35k×4096 = ~540 MB, fine on 80 GB

# Compile the embedder for faster query embedding (warm-up at first query)
try:
    VL_EMBEDDER._embedder.model = torch.compile(
        VL_EMBEDDER._embedder.model, mode='reduce-overhead', fullgraph=False
    )
    print('torch.compile applied to embedder')
except Exception as _ce:
    print(f'torch.compile on embedder skipped: {_ce}')

print(f'Total VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB / '
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
print(f'\nStage 3 done in {time.perf_counter()-_t0:.0f}s')


In [ ]:
# Stage 4: Build VibeScentEngine — Qwen3-VL unified stack
import base64, io, os, tempfile, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from vibescents.backend_app import create_app
from vibescents.schemas import RecommendRequest, RecommendResponse, FragranceRecommendation
from vibescents.structured_scorer import compute_structured_scores

_TOPK  = 8   # retrieve top-8, batch-rerank all 8 in ONE forward pass → top-3
_FINAL = 3

# ── Qwen3-VL-Reranker-8B canonical instruction format ────────────────────
# See: https://huggingface.co/Qwen/Qwen3-VL-Reranker-8B
# The <think>\n\n</think>\n\n suffix is MANDATORY — it primes the model's
# internal chain-of-thought before emitting the yes/no decision token.
_RERANKER_SYS = (
    'Judge whether the Document meets the requirements based on the Query and '
    'the Instruct provided. Note that the answer can only be "yes" or "no".\'
)
_RERANKER_INSTRUCTION = (
    'Given an outfit image and occasion context, assess whether the fragrance '
    'candidate is a strong match. Consider: formality, mood, season, and scent-aesthetic harmony.'
)


def _embed_query(context_str, image_bytes):
    if image_bytes:
        import tempfile as _tf_mod
        with _tf_mod.NamedTemporaryFile(suffix='.jpg', delete=False) as _tf:
            _tf.write(image_bytes)
            _tmp = _tf.name
        try:
            vec = VL_EMBEDDER.embed_multimodal_query(text=context_str, image_path=_tmp)
        finally:
            os.unlink(_tmp)
    else:
        vec = VL_EMBEDDER.embed_multimodal_documents([context_str])
    vec = vec.astype('float32').reshape(1, -1)
    vec /= (np.linalg.norm(vec, axis=1, keepdims=True) + 1e-9)
    return vec


@torch.no_grad()
def _rerank_batch(context_str, frag_texts, image_bytes):
    """Single batched forward pass for all candidates.

    Returns list[float] of P(yes) scores, same length as frag_texts.
    Uses log_softmax over the (no, yes) logit pair for numerical stability.
    """
    from qwen_vl_utils import process_vision_info
    import base64 as _b64

    _img_b64 = _b64.b64encode(image_bytes).decode() if image_bytes else None
    texts_for_proc, all_messages = [], []

    for frag_text in frag_texts:
        user_parts = []
        if _img_b64:
            user_parts.append({
                'type': 'image',
                'image': f'data:image/jpeg;base64,{_img_b64}',
            })
        # Canonical Qwen3-Reranker instruction layout
        user_parts.append({
            'type': 'text',
            'text': (
                f'<Instruct>: {_RERANKER_INSTRUCTION}\n'
                f'<Query>: {context_str}\n'
                f'<Document>: {frag_text}'
            ),
        })
        messages = [
            {'role': 'system', 'content': _RERANKER_SYS},
            {'role': 'user',   'content': user_parts},
        ]
        text = VL_RERANKER_PROC.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Mandatory CoT suffix — the model scores yes/no AFTER this token sequence
        text += '<think>\n\n</think>\n\n'
        texts_for_proc.append(text)
        all_messages.append(messages)

    # Process image once — all candidates share the same outfit image
    if _img_b64:
        _single_img_msg = [{'role': 'user', 'content': [{'type': 'image', 'image': f'data:image/jpeg;base64,{_img_b64}'}]}]
        _img_inputs_one, _ = process_vision_info(_single_img_msg)
        image_inputs = _img_inputs_one * len(frag_texts)  # same PIL object, avoids N-1 extra decodes
    else:
        image_inputs = []

    inputs = VL_RERANKER_PROC(
        text=texts_for_proc,
        images=image_inputs if image_inputs else None,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,  # 205-token typical budget; 512 gives 4x faster attention vs 1024
    ).to('cuda', non_blocking=True)

    out = VL_RERANKER_MODEL(**inputs)
    last_logits = out.logits[:, -1, :]  # (batch, vocab)

    # log_softmax over (no, yes) pair — numerically stable, avoids exp overflow
    no_l  = last_logits[:, _NO_ID]  if _NO_ID  >= 0 else torch.zeros(last_logits.shape[0], device=last_logits.device)
    yes_l = last_logits[:, _YES_ID] if _YES_ID >= 0 else torch.zeros(last_logits.shape[0], device=last_logits.device)
    p_yes = F.log_softmax(torch.stack([no_l, yes_l], dim=1), dim=1)[:, 1].exp()
    return p_yes.tolist()


def _build_frag_text(row):
    parts = []
    for col in ('name', 'brand', 'top_notes', 'middle_notes', 'base_notes',
                'main_accords', 'vibe_sentence', 'likely_occasion'):
        v = str(row.get(col, '') or '')
        if v and v.lower() not in ('nan', 'none', ''):
            parts.append(v)
    return ' | '.join(parts)[:400]


def _context_str(ctx):
    parts = [p for p in [ctx.eventType, ctx.timeOfDay, ctx.mood,
                         getattr(ctx, 'customNotes', None)] if p]
    return ', '.join(parts) if parts else 'general occasion'


class VibeScentEngine:
    """Qwen3-VL unified stack: embed → cosine retrieve → VL batch-rerank."""

    def __init__(self, *, fragrance_df, corpus_embeddings, locked_cache=None):
        self._df     = fragrance_df.reset_index(drop=True)
        self._corpus = corpus_embeddings  # already L2-normed
        self._cache  = locked_cache or {}

    def recommend(self, *, request, image_bytes):
        t0 = time.perf_counter()
        ctx = request.context
        # Cache key is context-only (image not included) — intentional for the demo:
        # locked responses cover the 5 canonical context combinations.
        cache_key = f'{ctx.eventType}|{ctx.timeOfDay}|{ctx.mood}'
        if cache_key in self._cache:
            return RecommendResponse(
                recommendations=[
                    FragranceRecommendation(**r)
                    for r in self._cache[cache_key]['recommendations']
                ]
            )

        ctx_str = _context_str(ctx)

        # ── Retrieval — GPU matmul (corpus pre-loaded in Stage 3) ─────
        q_vec = _embed_query(ctx_str, image_bytes)        # (1, D)
        q_gpu = torch.from_numpy(q_vec).to('cuda', non_blocking=True)
        scores = (CORPUS_EMBEDDINGS_GPU @ q_gpu.T).squeeze(1).cpu().numpy()  # (N,)

        struct_scores = compute_structured_scores(ctx, self._df)
        _lo, _hi = struct_scores.min(), struct_scores.max()
        struct_norm = (struct_scores - _lo) / max(_hi - _lo, 1e-9)
        fused = 0.85 * scores + 0.15 * struct_norm

        top_idx = np.argpartition(-fused, _TOPK)[:_TOPK]
        top_idx = top_idx[np.argsort(-fused[top_idx])]

        print(f'  Retrieval done ({time.perf_counter()-t0:.1f}s). '
              f'Top: {self._df.iloc[top_idx[0]].get("name", "?")}')

        # ── Batch rerank: one forward pass for all _TOPK candidates ───
        frag_texts = [_build_frag_text(self._df.iloc[int(i)]) for i in top_idx]
        try:
            rerank_scores = _rerank_batch(ctx_str, frag_texts, image_bytes)
        except Exception as _e:
            print(f'  Batch reranker failed: {_e} — falling back to fusion scores')
            rerank_scores = [float(fused[int(i)]) for i in top_idx]

        _ranked = sorted(zip(top_idx, rerank_scores),
                         key=lambda x: x[1], reverse=True)
        final_top = _ranked[:_FINAL]

        print(f'  Reranking done ({time.perf_counter()-t0:.1f}s). '
              f'Final top: {self._df.iloc[final_top[0][0]].get("name", "?")}')

        # ── Build response ────────────────────────────────────────────
        recs = []
        for rank, (idx, rr_score) in enumerate(final_top, start=1):
            row   = self._df.iloc[int(idx)]
            name  = str(row.get('name',  f'Fragrance #{idx}'))
            house = str(row.get('brand', 'Unknown'))
            notes = []
            for col in ('top_notes', 'middle_notes', 'base_notes', 'main_accords'):
                v = str(row.get(col, '') or '')
                if v and v.lower() not in ('nan', 'none', ''):
                    notes += [n.strip() for n in v.split(',') if n.strip()]
            notes = list(dict.fromkeys(notes))[:8]
            vibe = str(row.get('vibe_sentence', '') or '')
            if vibe.lower() in ('nan', 'none', ''): vibe = ''
            reasoning = vibe or f'Matched for {ctx_str} (score: {rr_score:.3f})'
            occ_lbl = str(row.get('likely_occasion', ctx_str) or ctx_str)
            if occ_lbl.lower() in ('nan', 'none', ''): occ_lbl = ctx_str
            recs.append(FragranceRecommendation(
                rank=rank, name=name, house=house,
                score=round(rr_score, 4),
                notes=notes, reasoning=reasoning, occasion=occ_lbl,
            ))

        print(f'  Total latency: {time.perf_counter()-t0:.1f}s')
        return RecommendResponse(recommendations=recs)


ENGINE = VibeScentEngine(
    fragrance_df=FRAGRANCE_DF,
    corpus_embeddings=CORPUS_EMBEDDINGS,
)
print(f'VibeScentEngine (w5v1) built. corpus={len(FRAGRANCE_DF)} fragrances  '
      f'[batch reranker, _TOPK={_TOPK}]')


In [ ]:
# Stage 5: Start FastAPI server
import threading, time, httpx, nest_asyncio, asyncio
nest_asyncio.apply()
import uvicorn
from vibescents.backend_app import create_app

_engine = globals().get('ENGINE')
if _engine is None:
    raise RuntimeError('ENGINE not defined. Run Stage 4 first.')

APP = create_app(engine=_engine)
_cfg = uvicorn.Config(app=APP, host=FASTAPI_HOST, port=FASTAPI_PORT,
                      log_level='warning', loop='asyncio', access_log=False)
_srv = uvicorn.Server(config=_cfg)

def _run(): asyncio.run(_srv.serve())
threading.Thread(target=_run, daemon=True, name='uvicorn').start()

_dl = time.time() + 30
while time.time() < _dl:
    try:
        if httpx.get(f'{_BASE_URL}/healthz', timeout=2.0).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(0.3)
else:
    raise RuntimeError('FastAPI did not start in 30s')

print(f'FastAPI ready at {_BASE_URL}')
print(f'Health: {httpx.get(f"{_BASE_URL}/healthz").json()}')

In [ ]:
# Stage 6: Dual Tunnel — Cloudflare primary + ngrok backup
import re, subprocess, time
CLOUDFLARE_URL = None; NGROK_URL = None

print('Starting Cloudflare Tunnel ...')
_cf = None
try:
    _cf = subprocess.Popen(
        [CLOUDFLARED_BIN, 'tunnel', '--url', f'http://localhost:{FASTAPI_PORT}', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    import atexit; atexit.register(lambda: _cf.kill() if _cf.poll() is None else None)
except Exception as _e:
    print(f'WARNING: cloudflared failed: {_e}')

_pat = re.compile(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com')
_dl  = time.time() + 60
while _cf is not None and time.time() < _dl:
    line = _cf.stdout.readline()
    if not line: time.sleep(0.2); continue
    m = _pat.search(line)
    if m: CLOUDFLARE_URL = m.group(0); break

if CLOUDFLARE_URL: print(f'Cloudflare: {CLOUDFLARE_URL}')
else: print('WARNING: Cloudflare URL not found')

print('Starting ngrok backup ...')
try:
    from pyngrok import ngrok as _ng
    _tun = _ng.connect(FASTAPI_PORT)
    NGROK_URL = _tun.public_url
    if NGROK_URL.startswith('http://'): NGROK_URL = 'https://' + NGROK_URL[7:]
    print(f'ngrok: {NGROK_URL}')
except Exception as _e:
    print(f'ngrok unavailable: {_e}')

ACTIVE_BACKEND_URL = CLOUDFLARE_URL or NGROK_URL
print('=' * 70)
print(f'PRIMARY (Cloudflare): {CLOUDFLARE_URL or "NOT AVAILABLE"}')
print(f'BACKUP  (ngrok):      {NGROK_URL or "NOT AVAILABLE"}')
print(f'\nSet in frontend .env.local:')
print(f'  NEXT_PUBLIC_BACKEND_URL={ACTIVE_BACKEND_URL or "<paste URL>"}')
print('=' * 70)

In [ ]:
# Stage 7: Locked demo cache — 5 context-only fallback responses
# NOTE: These are generated with a 1×1 white placeholder image, NOT a real outfit.
# They serve as tunnel-down fallbacks only; context matching still works.
# For outfit-driven accuracy, the live pipeline (Stages 1–6) must be running.
import json, os, time, httpx
from tqdm.auto import tqdm

_WHITE_PNG_B64 = (
    'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk'
    '+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg=='
)

LOCKED_CASES = [
    {'key': 'Gala|Evening|Mysterious',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Gala', 'timeOfDay': 'Evening', 'mood': 'Mysterious', 'customNotes': None}}},
    {'key': 'Business|Morning|Bold',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Business', 'timeOfDay': 'Morning', 'mood': 'Bold', 'customNotes': None}}},
    {'key': 'Date Night|Evening|Warm',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Date Night', 'timeOfDay': 'Evening', 'mood': 'Warm', 'customNotes': None}}},
    {'key': 'Casual|Afternoon|Fresh',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Casual', 'timeOfDay': 'Afternoon', 'mood': 'Fresh', 'customNotes': None}}},
    {'key': 'Wedding|Evening|Subtle',
     'request': {'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
                 'context': {'eventType': 'Wedding', 'timeOfDay': 'Evening', 'mood': 'Subtle', 'customNotes': None}}},
]

local_cache_path = os.path.join(REPO_DIR, 'artifacts', 'week5', 'locked_responses.json')
os.makedirs(os.path.dirname(local_cache_path), exist_ok=True)
locked_responses = {}

_pb = tqdm(LOCKED_CASES, desc='Generating locked responses', ncols=90, unit='case')
for case in _pb:
    key = case['key']; _pb.set_postfix_str(key)
    try:
        resp = httpx.post(f'{_BASE_URL}/recommend', json=case['request'], timeout=180.0)
        resp.raise_for_status()
        locked_responses[key] = resp.json()
        _n = len(locked_responses[key].get('recommendations', []))
        _top = locked_responses[key]['recommendations'][0]['name'] if _n else '?'
        print(f"  '{key}' -> {_n} recs | top: {_top}")
    except Exception as _e:
        print(f"  WARNING: '{key}' failed: {_e}")
_pb.close()

ENGINE._cache.update(locked_responses)
print(f'Injected {len(locked_responses)} responses into engine cache')

_payload = json.dumps(locked_responses, indent=2)
with open(local_cache_path, 'w') as _lf: _lf.write(_payload)
try:
    with open(os.path.join(W5_ARTIFACTS, 'locked_responses.json'), 'w') as _df2: _df2.write(_payload)
    print('Saved to local + Drive')
except Exception: print('Saved to local only')

In [ ]:
# Stage 8: Frontend Integration — set backend URL in .env.local
# The frontend already reads ML_BACKEND_URL from server-side environment.
# No code changes needed — just set the env variable.
import os

_backend = globals().get('ACTIVE_BACKEND_URL') or globals().get('CLOUDFLARE_URL') or globals().get('NGROK_URL', '')

print('=' * 70)
print('  FRONTEND INTEGRATION')
print('=' * 70)
print()
print('Add to your frontend .env.local (or Vercel env vars):')
print()
print(f'  ML_BACKEND_URL={_backend or "<paste Cloudflare or ngrok URL here>"}')
print()
print('The Next.js route (/api/recommend) already proxies to ML_BACKEND_URL.')
print('No file changes are needed in the frontend repo.')
print()
print('=' * 70)

# Save the URL to a text file for quick reference
_ref_path = os.path.join(REPO_DIR, 'artifacts', 'week5', 'backend_url.txt')
os.makedirs(os.path.dirname(_ref_path), exist_ok=True)
with open(_ref_path, 'w') as _uf:
    _uf.write(f'ML_BACKEND_URL={_backend}\n')
try:
    with open(os.path.join(W5_ARTIFACTS, 'backend_url.txt'), 'w') as _uf2:
        _uf2.write(f'ML_BACKEND_URL={_backend}\n')
    print(f'URL saved: local + Drive')
except Exception:
    print(f'URL saved: local only')


In [ ]:
# Stage 9: Smoke Test
import httpx

_PAYLOAD = {
    'image': _WHITE_PNG_B64, 'mimeType': 'image/png',
    'context': {'eventType': 'Gala', 'timeOfDay': 'Evening', 'mood': 'Mysterious', 'customNotes': None},
}
_REQUIRED = {'rank', 'name', 'house', 'score', 'notes', 'reasoning', 'occasion'}

def _validate(data, url):
    recs = data.get('recommendations', [])
    assert recs, f'Empty recommendations from {url}'
    for i, r in enumerate(recs):
        miss = _REQUIRED - set(r.keys())
        assert not miss, f'rec[{i}] missing fields: {miss}'
    print(f'  PASS ({url}) — {len(recs)} recs')
    for r in recs: print(f'    #{r["rank"]} {r["name"]} | {r["house"]} | score={r["score"]:.4f}')

_ok = True
print('Test 1: Local server')
try:
    _r = httpx.post(f'{_BASE_URL}/recommend', json=_PAYLOAD, timeout=180.0)
    _r.raise_for_status(); _validate(_r.json(), _BASE_URL)
except Exception as _e: print(f'  FAIL: {_e}'); _ok = False

if CLOUDFLARE_URL:
    print(f'Test 2: Cloudflare')
    try:
        _r2 = httpx.post(f'{CLOUDFLARE_URL}/recommend', json=_PAYLOAD, timeout=60.0, follow_redirects=True)
        _r2.raise_for_status(); _validate(_r2.json(), CLOUDFLARE_URL)
    except Exception as _e2: print(f'  WARNING: {_e2}')

print(f'Health: {httpx.get(f"{_BASE_URL}/healthz").json()}')
print('All smoke tests PASSED' if _ok else 'WARNING: some tests failed')

In [ ]:
# Stage 10: Pre-Demo Runbook
import os

_D = '=' * 70
print(_D); print('  VIBESCENT WEEK 5 — QWEN3-VL UNIFIED STACK — PRE-DEMO RUNBOOK'); print(_D)
print('''
BEFORE PRESENTING
-----------------
1. Select Blackwell 6000 / A100 80 GB runtime
   Runtime > Change runtime type > GPU: A100/Blackwell

2. Stage 0: SKIP if qwen3vl_corpus/embeddings.npy already on Drive
   Run Stage 0 only if corpus embeddings were deleted or this is a fresh account

3. Run all remaining stages top-to-bottom (~10-15 min first run)
   Stages 1-4 load models and build engine
   Stages 5-6 start server and tunnels
   Stage 7 pre-generates 5 locked responses (~3 min — reranker runs 100 pairs)

4. Copy PRIMARY tunnel URL to Darren frontend .env.local

5. Smoke test Stage 9 confirms pipeline end-to-end

NO API KEYS NEEDED AT INFERENCE — all models run locally.
HF_TOKEN needed only for first-time model download (cached to Drive after).
''')

print('TUNNEL URLS')
print('-' * 40)
print(f'  PRIMARY: {globals().get("CLOUDFLARE_URL", "NOT SET")}')
print(f'  BACKUP:  {globals().get("NGROK_URL",     "NOT SET")}')
print()

print('IF TUNNEL DROPS')
print('-' * 40)
print('A: Switch to ngrok backup')
print('B: Re-run Stage 6 only — new tunnel in 30s')
print('C: Context-only fallback — 5 locked responses (no outfit image):')
for _c in globals().get('LOCKED_CASES', []):
    print(f'   * {_c["key"]}')
print()

print('ARCHITECTURE TALKING POINTS')
print('-' * 40)
print('  Embedding: Qwen3-VL-Embedding-8B (MMEB-V2 #1, 77.8%) — direct image embed')
print('  Reranker:  Qwen3-VL-Reranker-8B — sees outfit image during ranking')
print('  No API calls at inference — fully local on 80 GB GPU')
print('  Vector space is consistent: same model for corpus and query')
print()

print('PIPELINE SUMMARY')
print('-' * 40)
print(f'  version    : {PIPELINE_VERSION}')
print(f'  GPU tier   : {globals().get("GPU_TIER", "unknown")}')
print(f'  corpus     : {len(globals().get("FRAGRANCE_DF", []))} fragrances')
print(f'  emb model  : Qwen/Qwen3-VL-Embedding-8B')
print(f'  reranker   : Qwen/Qwen3-VL-Reranker-8B')
print(f'  API keys   : none at inference')
print()
print(_D); print('  BACKEND IS READY. GOOD LUCK!'); print(_D)